## Cell 1: Start Spark

In [1]:
import os
import subprocess
from pathlib import Path

os.environ["JAVA_HOME"] = subprocess.check_output(
  ["/usr/libexec/java_home", "-v", "17"]).decode().strip()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
RAW_DIR = PROJECT_ROOT/ "data" / "raw"

spark = (SparkSession.builder
         .appName("globalpartners-practice")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate()
         )
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/14 05:13:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Cell 2, exercise 1: Read the data (and fall into a trap)

In [2]:
items_raw = spark.read.csv(str(RAW_DIR / "order_items.csv"), header=True,  multiLine=True, escape='"')
items_raw = items_raw.toDF(*[c.strip().lower() for c in items_raw.columns])

print("rows:", items_raw.count())
items_raw.printSchema()

items_raw.where(~F.col("creation_time_utc").rlike(r"^\d{4}-\d{2}-\d{2}T"))\
  .select("app_name", "restaurant_id", "creation_time_utc").show(5)
  
items_raw.select("app_name", "restaurant_id", "creation_time_utc").show(100)


rows: 203519
root
 |-- app_name: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- creation_time_utc: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- printed_card_number: string (nullable = true)
 |-- is_loyalty: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- lineitem_id: string (nullable = true)
 |-- item_category: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- item_price: string (nullable = true)
 |-- item_quantity: string (nullable = true)

+--------+-------------+-----------------+
|app_name|restaurant_id|creation_time_utc|
+--------+-------------+-----------------+
+--------+-------------+-----------------+

+--------------------+--------------------+--------------------+
|            app_name|       restaurant_id|   creation_time_utc|
+--------------------+--------------------+--------------------+
|       Alltown Fresh|63bc98a7519adc105...|2023-03-

## Cell 3, exercise 2: Fix the types

In [3]:
TS_FORMAT = "yyyy-MM-dd'T'HH:mm:ss[.SSS]X"      # [.SSS] = optional 1-3 digit fraction

items = (items_raw
    .withColumn("order_ts_utc", F.to_timestamp("creation_time_utc", TS_FORMAT))
    .withColumn("business_date", F.to_date(F.from_utc_timestamp("order_ts_utc", "America/New_York")))
    .withColumn("item_price", F.col("item_price").cast("decimal(10,2)"))
    .withColumn("item_quantity", F.col("item_quantity").cast("int"))
    .withColumn("is_loyalty", F.col("is_loyalty") == "TRUE"))

print("unparsed timestamps:", items.where(F.col("order_ts_utc").isNull()).count())
items.select("creation_time_utc", "order_ts_utc", "business_date", "item_price", "is_loyalty").show(5, truncate=False)

unparsed timestamps: 0
+------------------------+-----------------------+-------------+----------+----------+
|creation_time_utc       |order_ts_utc           |business_date|item_price|is_loyalty|
+------------------------+-----------------------+-------------+----------+----------+
|2023-03-08T11:03:32.223Z|2023-03-08 11:03:32.223|2023-03-08   |5.99      |false     |
|2023-05-13T12:45:00.475Z|2023-05-13 12:45:00.475|2023-05-13   |7.99      |false     |
|2023-02-03T07:15:28.519Z|2023-02-03 07:15:28.519|2023-02-03   |5.99      |false     |
|2023-08-17T16:18:02.231Z|2023-08-17 16:18:02.231|2023-08-17   |11.99     |false     |
|2023-08-20T16:02:12.518Z|2023-08-20 16:02:12.518|2023-08-20   |5.99      |false     |
+------------------------+-----------------------+-------------+----------+----------+
only showing top 5 rows


## exercise 3: Filter to the 2023 scope

In [4]:
scope = items.where(
  (F.year('business_date')==2023) &
  (F.col("app_name") != "Alltown Fresh - DEVELOPMENT")
)

print("lines      :", scope.count())
print("orders     :", scope.select("order_id").distinct().count())
print("locations  :", scope.select("restaurant_id").distinct().count())

lines      : 79965
orders     : 52015
locations  : 20


## groupBy, revenue per location

In [5]:
by_location = (scope
               .groupBy("restaurant_id")
               .agg(
                 F.countDistinct("order_id").alias("orders"),
                 F.sum("item_price").alias("item_revenue"),
               )
               .withColumn("avg_order_value", F.round(F.col("item_revenue") / F.col("orders"), 2))
               .orderBy(F.desc("item_revenue")))

by_location.show(5, truncate=False)
by_location.orderBy("item_revenue").show(1, truncate=False)





+------------------------+------+------------+---------------+
|restaurant_id           |orders|item_revenue|avg_order_value|
+------------------------+------+------------+---------------+
|5fece84aadb0d51509c36f22|9354  |130057.32   |13.90          |
|63bc98a7519adc105105a990|5733  |83074.62    |14.49          |
|62f2ce9824813746ce6f5140|5250  |72729.18    |13.85          |
|5e7e35ec902ad5ac017b242a|4920  |66495.80    |13.52          |
|6351b7d1a036663f05239b60|4779  |62400.42    |13.06          |
+------------------------+------+------------+---------------+
only showing top 5 rows
+------------------------+------+------------+---------------+
|restaurant_id           |orders|item_revenue|avg_order_value|
+------------------------+------+------------+---------------+
|61bccd6a7b92a75dc16baf4e|3     |22.46       |7.49           |
+------------------------+------+------------+---------------+
only showing top 1 row


## exercise 5: the "order counts don't add up" trap

In [6]:
by_category = (
  scope
  .groupBy("restaurant_id", "item_category")
  .agg(F.countDistinct("order_id").alias("orders"))
)

print("orders added up across categories:", by_category.agg(F.sum("orders")).first()[0])
print("real number of orders            :", scope.select("order_id").distinct().count())

orders added up across categories: 67215
real number of orders            : 52015


## exercise 6: join, full revenue with options

In [7]:
options = spark.read.csv(str(RAW_DIR / "order_item_options.csv"), header=True, multiLine=True, escape='"')
options = (options.toDF(*[c.strip().lower() for c in options.columns]))

option_totals = (options
                  .groupBy("order_id", "lineitem_id")
                  .agg(F.sum("option_price").alias("option_price_sum")))

lines = (scope
         .join(option_totals, on=["order_id", "lineitem_id"], how="left")
         .withColumn("option_price_sum", F.coalesce("option_price_sum", F.lit(0)))
         .withColumn("line_revenue", F.col("item_price") + F.col("option_price_sum") * F.col("item_quantity")))

inner = scope.join(option_totals, on=["order_id", "lineitem_id"], how="inner")

print("lines with LEFT join   :", lines.count())
print("lines with INNER join  :", inner.count())
print("total_revenue          :", lines.agg(F.sum("line_revenue")).first()[0])



lines with LEFT join   : 79965
lines with INNER join  : 40932
total_revenue          : 746223.8599995069


In [8]:
cols = [" Name", "PRICE ", "Qty"]
print(*[c.strip() for c in cols])

Name PRICE Qty


In [9]:
from pyspark.sql import Window

NON_CUSTOMER_IDS = ["5ece77fe902ad501337b23fd"]

customer_orders = (lines
    .where(F.col("user_id").isNotNull() & ~F.col("user_id").isin(NON_CUSTOMER_IDS))
    .groupBy("user_id", "order_id", "order_ts_utc")
    .agg(F.sum("line_revenue").alias("order_revenue")))

running_total = (Window
    .partitionBy("user_id")                                    # each customer separately
    .orderBy("order_ts_utc")                                   # oldest order first
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)) # from the first order up to this one

customer_orders = customer_orders.withColumn("clv_to_date", F.sum("order_revenue").over(running_total))

customer_orders.where(F.col("user_id") == "642d6946c9dae475e80a9f4e").orderBy("order_ts_utc").show(truncate=False)
print("customers:", customer_orders.select("user_id").distinct().count())

+------------------------+------------------------+-----------------------+------------------+-----------+
|user_id                 |order_id                |order_ts_utc           |order_revenue     |clv_to_date|
+------------------------+------------------------+-----------------------+------------------+-----------+
|642d6946c9dae475e80a9f4e|64591f67da7dc39d7a08193f|2023-05-08 16:12:23.01 |12.99             |12.99      |
|642d6946c9dae475e80a9f4e|6499c23e315790dc580214d7|2023-06-26 16:52:14.335|10.99             |23.98      |
|642d6946c9dae475e80a9f4e|6499caa91480079ff90c8517|2023-06-26 17:28:09.613|10.99             |34.97      |
|642d6946c9dae475e80a9f4e|64d3b041864d59b81f0df255|2023-08-09 15:26:57.377|16.990000000000002|51.96      |
|642d6946c9dae475e80a9f4e|64d3ba3affecd7abfa0a1924|2023-08-09 16:09:30.946|11.99             |63.95      |
|642d6946c9dae475e80a9f4e|653698325330f62fcf0849a0|2023-10-23 15:58:42.498|10.99             |74.94      |
+------------------------+-----------

In [ ]:
clv = customer_orders.groupBy("user_id").agg(F.sum("order_revenue").alias("clv"))

by_clv = Window.orderBy("clv")
tiers = (clv
         .withColumn("clv_cume_dist", F.cume_dist().over(by_clv))
         .withColumn("clv_tier",
                     F.when(F.col("clv_cume_dist")>0.8, "High")
                     .when(F.col("clv_cume_dist")<=0.2, "Low")
                     .otherwise("Medium"))
         )
(tiers.groupBy("clv_tier")
 .agg(F.count("*").alias("customers"), F.min("clv").alias("min_clv"), F.max("clv").alias("max_clv"))
 .orderBy("min_clv")
 .show())

+--------+---------+-----------------+------------------+
|clv_tier|customers|          min_clv|           max_clv|
+--------+---------+-----------------+------------------+
|     Low|     1805|             1.49|             10.98|
|  Medium|     6604|            10.99|             75.92|
|    High|     2103|75.92999999999999|4051.3399999999992|
+--------+---------+-----------------+------------------+



In [ ]:
daily = (lines
         .where(F.col("user_id").isNotNull() & ~F.col("user_id").isin(NON_CUSTOMER_IDS))
         .groupBy("user_id", "business_date")
         .agg(
           F.countDistinct("order_id").alias("orders_on_day"),
           F.sum("line_revenue").alias("revenue_on_day")
         ))

print("customer-days with orders:", daily.count())




customer-days with orders: 43993


In [ ]:
END_DATE = "2023-12-31"

first_orders = daily.groupBy("user_id").agg(F.min("business_date").alias("first_order_date"))
spine = first_orders.withColumn(
  "snapshot_date",
  F.explode(F.sequence("first_order_date", F.to_date(F.lit(END_DATE))))
)

snapshot= (spine
           .join(daily.withColumnRenamed("business_date", "snapshot_date"),
                 on=["user_id", "snapshot_date"], how="left")
           .withColumn("orders_on_day", F.coalesce("orders_on_day", F.lit(0)))
           .withColumn("revenue_on_day", F.coalesce("revenue_on_day", F.lit(0)))
           .withColumn("day_nun", F.datediff("snapshot_date", F.to_date(F.lit("1970-01-01")))))


+------------------------+-------------+----------------+-------------+--------------+
|user_id                 |snapshot_date|first_order_date|orders_on_day|revenue_on_day|
+------------------------+-------------+----------------+-------------+--------------+
|648b05fe412e0c0309088086|2023-06-15   |2023-06-15      |1            |7.99          |
|648b05fe412e0c0309088086|2023-06-16   |2023-06-15      |NULL         |NULL          |
|648b05fe412e0c0309088086|2023-06-17   |2023-06-15      |NULL         |NULL          |
|648b05fe412e0c0309088086|2023-06-18   |2023-06-15      |NULL         |NULL          |
|648b05fe412e0c0309088086|2023-06-19   |2023-06-15      |NULL         |NULL          |
|648b05fe412e0c0309088086|2023-06-20   |2023-06-15      |NULL         |NULL          |
|648b05fe412e0c0309088086|2023-06-21   |2023-06-15      |NULL         |NULL          |
|648b05fe412e0c0309088086|2023-06-22   |2023-06-15      |NULL         |NULL          |
|648b05fe412e0c0309088086|2023-06-23   |202